# 00_config — 공통 설정 (담당: core)

- **이 파일은 core 담당만 고칩니다.** 팀원은 받은 그대로 쓰고, 바뀌면 새 파일로 교체합니다. 불러올 때 출력되는 해시가 공지된 해시와 같은지 확인하세요.
- 모든 노트북은 `notebooks/` 폴더 안에서 실행합니다. 첫 칸은 아래 세 줄입니다. 경로는 자동으로 잡힙니다.
  ```python
  SKIP_LIB_TESTS = True
  %run -i ./00_config.ipynb
  %run -i ./lib_prep.ipynb
  ```
- Colab은 프로젝트 폴더를 올린 뒤 첫 칸 맨 위에 `%cd /content/lendingclub/notebooks`를 둡니다.
- **미정인 것은 모두 `pending()`으로 감싸 둡니다.** 개발 중에는 임시값으로 돌고, `STAGE = "final"`이면 미정이 하나라도 남았을 때 멈춥니다. 불러올 때 미정 목록이 출력됩니다.
- `pending()`의 세 번째 인자는 쟁점과 정할 사람입니다. `[이견]`, `[근거 없음]` 표시는 결정 문서(Drive `2026-11-25/decisions/`)와 같은 뜻입니다.
- **표시용 미정**(`SIGN_TABLE`, `RECOVERY_LAG`, `SENS_SETS`)은 실제 값이 `lib_prep`에 있습니다. 여기 값을 바꿔도 코드에 반영되지 않습니다. 확정하면 `lib_prep`을 고치고 이 줄을 지웁니다.
- 미정을 확정할 때는 `pending(...)`을 값으로 바꾸고, 옆 주석에 날짜와 근거를 적습니다. 해시가 바뀌므로 팀에 공지합니다.

In [9]:
# 1. 경로: 내려받은 프로젝트 폴더 기준
# 보통은 None으로 두면 자동으로 찾습니다. 자동으로 못 찾을 때만 core 담당이 경로를 넣습니다(팀원은 고치지 않음).
PROJECT_DIR = None   # 예: r"C:/Users/이름/Downloads/lendingclub"  또는  "/Users/이름/Downloads/lendingclub"

import os, json, hashlib, platform
from importlib import metadata
from pathlib import Path
import numpy as np

def find_root():
    """PROJECT_DIR → LC_ROOT 환경변수 → 현재 폴더에서 위로 올라가며 notebooks/00_config.ipynb가 있는 폴더."""
    for given in (PROJECT_DIR, os.environ.get("LC_ROOT")):
        if given:
            p = Path(given).expanduser().resolve()
            assert (p / "notebooks" / "00_config.ipynb").exists(), f"이 경로에 notebooks/00_config.ipynb가 없음: {p}"
            return p
    here = Path.cwd().resolve()
    for d in (here, *here.parents):
        if (d / "notebooks" / "00_config.ipynb").exists():
            return d
    raise FileNotFoundError("프로젝트 폴더를 못 찾음. notebooks/ 안에서 실행하세요.")

ROOT = find_root()
RAW_DIR, NB_DIR, DATA_DIR, CONTRACT_DIR, OUT_DIR, REPORT_DIR = (
    ROOT / d for d in ("raw", "notebooks", "data", "contract", "outputs", "report"))
TRAIN_CSV    = RAW_DIR / "lending_club_2020_train.csv"      # 01_prep(core 담당)만 필요
TREASURY_CSV = RAW_DIR / "us_treasury_3y_5y_monthly.csv"
# Test 파일 경로는 두지 않는다. 사전 고정 뒤 05_test에서만 지정한다.

In [10]:
# 2. 실험 설정
PENDING = {}
def pending(name, dev_value, note):
    """미결 값: 개발 중에는 dev_value로 돌고, 이름과 결정 방법을 PENDING에 남긴다."""
    PENDING[name] = note
    return dev_value

CONFIG_VERSION = "2026-11-26"
STAGE = "dev"              # "dev" | "final"
SEED  = 42

# 표본: load_and_prepare 인자와 01_prep 검증값 (01_eda_audit 2026-11-24 실행 결과)
SAMPLE = dict(mode="screened_t0", start_year=2013, maturity_cutoff_ym="2020-04")
EXPECT = dict(n_dev=666_195, default_rate=0.1626, n_raw_features=37)

# 전처리 (preprocess.py 2026-11-24 기본값). 희귀 범주 0.5%는 2026-11-25 확정
PREP = dict(dti_clip_q=0.995,          # 2026-11-26 확정(일정상 기본값 채택). preprocess.py 기본값, 근거 문헌 없음 → 한계에 기재
            rare_cat_min_freq=0.005)  # 2026-11-25 확정

# 표시용 미정: 실제 값은 lib_prep에 있음 (여기를 바꿔도 반영되지 않음)
SIGN_TABLE   = "FEATURE_SPEC 2026-11-24 원안 + emp_length 0"   # 2026-11-26 확정(core 담당). 근거 report/sign_check.csv (표시용)
RECOVERY_LAG = "lib_prep 기본값 5.0개월"   # 2026-11-26 확정(일정상 기본값 채택). 근거 문헌 없음 → 한계에 기재 (표시용)
SENS_SETS    = "core_t0(16개)만"           # 2026-11-26 확정(일정상 기본값 채택). 52개 세트 민감도는 범위에서 제외 → 한계·향후 과제 (표시용)

# 부트스트랩과 커트라인
B         = 100            # 2026-11-25 확정 (Steyerberg et al., 2001. 원문 쪽수는 담당자 확인 중)
B_TEST    = pending("B_TEST", 4_999, "규칙 확정: 4,999회가 10분 안이면 4,999, 아니면 1,999. eval 담당이 시간 측정 후 값만 기입")
RATES     = np.round(np.arange(0.05, 0.951, 0.01), 2)   # 2026-11-26 확정(일정상 기본값 채택). 격자 1%p
MIN_RATE  = 0.05             # 2026-11-26 확정(일정상 기본값 채택). with_treasury에서는 비구속(Dev OOB a* 0.41~0.58)
SR_MODE   = "with_treasury"  # 2026-11-25 확정: MIRR 초과수익률, 동일가중, 거절 대출=0을 포함한 OOB 전체로 SR 계산
                             # (Park et al. 2026 JDQS 정의와 수치상 동일). 'approved_only'는 a*가 하한에 붙어 참고 그림용
REF_MODEL = "xgb"          # 주 분석: ②의 a*를 세 모형 공통 승인율로
ALPHA     = 0.025          # 비열등 단측 → ΔSR 분포의 97.5% 분위수와 DELTA 비교
DELTA     = pending("DELTA", None, "Dev OOB의 ΔSR 분포와 경제적 의미로 core 담당이 결정")

# 모형
LOGIT_PARAMS = dict(C=np.inf, max_iter=2000)        # 비정규화 MLE: 페널티 끔. penalty=None은 sklearn 1.8부터 폐기 예정  # max_iter [검토]
XGB_FIXED    = dict(objective="binary:logistic", eval_metric="logloss", tree_method="hist",
                    random_state=SEED, n_jobs=-1)            # OOB로 early stopping 금지
XGB_PARAMS   = pending("XGB_PARAMS", dict(n_estimators=200, max_depth=4, learning_rate=0.05,
                                          subsample=0.8, colsample_bytree=0.8),
                       "xgb 담당 튜닝 결과로 교체")
MONO_HP_MODE = "same_as_xgb"   # 2026-11-26 확정(일정상 기본값 채택). ②와 같은 파라미터(통제 비교). 튜닝이 부호표를 기다리지 않음. own_tuning은 한계·향후 과제
# 튜닝: 부트스트랩 전 Dev 전체에서 1회 (순서는 확정, 아래 세부값은 모두 미정)
TUNE         = dict(           # 2026-11-26 확정(일정상 기본값 채택). 근거 문헌 없음 → 방법 절에 설정 그대로 기재
                   method="optuna_tpe", n_trials=30, sampler_seed=SEED, cv_folds=3,
                   metric="logloss", early_stopping_rounds=50, max_trees=2000,
                   fold_prep="refit_per_fold", final_trees="mean_best_iteration",
                   space=dict(max_depth=(2, 6), learning_rate=(0.02, 0.2, "log"),
                              min_child_weight=(1, 100, "log"), subsample=(0.6, 1.0),
                              colsample_bytree=(0.5, 1.0), reg_lambda=(0.1, 10, "log")))

# 6-C 설명 지표
EXPLAIN = dict(top_k=3, perturb_sd=0.25, viol_tol=1e-6, jaccard_pairs=200, rbo_p=0.9)  # 2026-11-26 확정(일정상 기본값 채택). 근거 문헌 없음 → 한계에 기재

# Test 평가 (Step 6)
CI_METHOD   = "percentile"   # 2026-11-26 확정(일정상 기본값 채택). Ledoit&Wolf(2008)는 studentized → 보고서에서 "LW처럼 부트스트랩" 정도로 인용
AUDIT_CASES = dict(n=3, pick="random", seed=SEED, where="커트라인 바로 밖 거절")  # 2026-11-26 확정(일정상 기본값 채택). 결과 보기 전 고정

# 견고성 분석 (같은 예측값으로 재계산만). 목록과 판정 규칙을 사전 등록
ROBUST = dict(weights=("equal", "loan_amnt"), returns=("mirr", "hpr"), rate_band=(0.35, 0.63),
              own_a_star=False, pd_cutoff=False,
              rule="세 견고성 모두에서 ΔSR 부호가 주 분석과 같으면 견고")  # 2026-11-26 확정(일정상 기본값 채택). 구간은 Dev OOB 평평 구간

if STAGE == "final":
    assert not PENDING, f"미결 항목이 남아 있음: {sorted(PENDING)}"


In [11]:
# 3. 기록용 함수
_CONFIG_KEYS = ["CONFIG_VERSION", "STAGE", "SEED", "SAMPLE", "PREP", "B", "B_TEST", "MIN_RATE",
                "SR_MODE", "REF_MODEL", "ALPHA", "DELTA", "LOGIT_PARAMS", "XGB_FIXED", "XGB_PARAMS",
                "MONO_HP_MODE", "TUNE", "EXPLAIN", "SIGN_TABLE", "RECOVERY_LAG", "SENS_SETS",
                "CI_METHOD", "AUDIT_CASES", "ROBUST"]

def config_dict():
    d = {k: globals()[k] for k in _CONFIG_KEYS}
    d["RATES"] = [float(RATES[0]), float(RATES[-1]), len(RATES)]
    return d

def config_fingerprint():
    """설정값 해시(12자). 결과 파일(summary.json)에 같이 남긴다."""
    s = json.dumps(config_dict(), sort_keys=True, ensure_ascii=False, default=str)
    return hashlib.sha256(s.encode()).hexdigest()[:12]

def run_meta():
    """summary.json에 넣을 공통 메타: 설정, 해시, 미결 목록, 패키지 버전."""
    def ver(p):
        try: return metadata.version(p)
        except metadata.PackageNotFoundError: return None
    return dict(config=config_dict(), fingerprint=config_fingerprint(), pending=sorted(PENDING),
                lib_prep=globals().get("LIB_PREP_HASH"), python=platform.python_version(),
                packages={p: ver(p) for p in ("numpy", "pandas", "scikit-learn", "xgboost", "shap", "pyarrow")})

def ensure_dirs():
    for d in (DATA_DIR, CONTRACT_DIR, OUT_DIR, REPORT_DIR):
        d.mkdir(parents=True, exist_ok=True)

ensure_dirs()
print(f"[config {CONFIG_VERSION}] 해시 {config_fingerprint()} | STAGE={STAGE} | "
      + (f"미결 {len(PENDING)}개 ({', '.join(PENDING)}), 임시값으로 실행 중" if PENDING else "미결 없음"))

[config 2026-11-26] 해시 a01f81ea67d1 | STAGE=dev | 미결 3개 (B_TEST, DELTA, XGB_PARAMS), 임시값으로 실행 중


In [12]:
# 4. 점검 (이 노트북을 직접 열었을 때만)
if not globals().get("SKIP_LIB_TESTS", False):
    print(f"ROOT={ROOT}")
    for k, v in PENDING.items():
        print(f"  미결 {k}: {v}")
    for p in (TREASURY_CSV, TRAIN_CSV, DATA_DIR / "dev_X_raw.parquet", DATA_DIR / "dev_meta.parquet",
              CONTRACT_DIR / "categories.json"):
        print("  OK  " if p.exists() else "  없음", p.relative_to(ROOT))
    leaked = [p.name for p in RAW_DIR.glob("*") if "test" in p.name.lower()]
    assert not leaked, f"raw/에 Test 파일이 있음: {leaked}. 사전 고정 전에는 두지 마세요."


ROOT=~/Desktop/서울대/KDT/통계/팀플/lendingclub
  미결 B_TEST: 규칙 확정: 4,999회가 10분 안이면 4,999, 아니면 1,999. eval 담당이 시간 측정 후 값만 기입
  미결 DELTA: Dev OOB의 ΔSR 분포와 경제적 의미로 core 담당이 결정
  미결 XGB_PARAMS: xgb 담당 튜닝 결과로 교체
  OK   raw/us_treasury_3y_5y_monthly.csv
  OK   raw/lending_club_2020_train.csv
  OK   data/dev_X_raw.parquet
  OK   data/dev_meta.parquet
  OK   contract/categories.json
